# 🧬 OBJ-1 KG Agent
## Environment: Python 3.8.6 · ipykernel

**Handles intents:** `DISEASE_TO_TARGET`, `DISEASE_TO_PATHWAY`, `TARGET_TO_PATHWAY`,
`DRUG_TO_TARGET`, `MOLECULE_TO_TARGET`, `MOLECULE_TO_PATHWAY`, `MOLECULE_TO_DISEASE`,
`DISEASE_PPI_NETWORK`, `DISEASE_PPI_HUBS`, `COMMON_DISEASE_DISEASE_PATHWAY`,
`DISEASE_DRUGGABLE_TARGETS`, `FULL_PIPELINE`

**Writes:** `shared/kg_output.json` → read by the Orchestrator and Obj-2

### ⚠️ Run this in the `ipykernel` (Python 3.8.6) kernel only

In [1]:
# Run once — install LangChain in this environment
%pip install langchain langchain-core langgraph rapidfuzz pyvis

Note: you may need to restart the kernel to use updated packages.


## ─── CELL A: All original Obj-1 imports (UNCHANGED) ───

In [2]:
# =========================
# Core Libraries
# =========================
import os, json, re, requests, operator
import pandas as pd
import numpy as np
import networkx as nx
from typing import TypedDict, Annotated, Any, Optional

# Bioinformatics
from Bio.PDB import PDBParser, PDBIO

# Molecular
from rdkit import Chem
#from rdkit.Chem import AllChem

# Subprocess
import subprocess

# Visualization
import py3Dmol
from pyvis.network import Network
from IPython.display import display, HTML

# LLM — Ollama package (available in ipykernel env)
import ollama

# LangChain — NEW additions
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, END

# Shared output directory
SHARED_DIR = os.path.join(os.getcwd(), "shared")
os.makedirs(SHARED_DIR, exist_ok=True)
KG_OUTPUT_FILE = os.path.join(SHARED_DIR, "kg_output.json")

print("✅ Obj-1 imports complete  |  Python 3.8.6 / ipykernel")

✅ Obj-1 imports complete  |  Python 3.8.6 / ipykernel


## ─── CELL B: All original Obj-1 functions (COMPLETELY UNCHANGED) ───

In [3]:
# ══════════════════════════════════════════════════════════════
# ORIGINAL OBJ-1 CODE — NOT A SINGLE LINE CHANGED
# Every function is exactly as it appears in your notebook.
# ══════════════════════════════════════════════════════════════

# ─── LLM ───
def call_ollama(model, system, user):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ],
        options={"temperature": 0}
    )
    return response["message"]["content"]


def safe_json_load(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
        raise ValueError("❌ No valid JSON found in LLM output")


def check_step(step_name, value):
    print(f"\n--- {step_name} ---")
    print(value)
    if value is None or value == "" or value == {}:
        raise RuntimeError(f"❌ {step_name} failed")


print("✅ Base utilities loaded")

✅ Base utilities loaded


In [4]:
# ─── KG ───
knowledge_graph = nx.read_gexf("data/knowledge_graph.gexf")
print("KG loaded:", knowledge_graph.number_of_nodes(), "nodes /", knowledge_graph.number_of_edges(), "edges")

KG loaded: 80756 nodes / 241886 edges


In [5]:
# ─── Query Understanding ───
def understand_query(query):
    system = "You are a biomedical query interpreter."
    user = f"""
Explain what the user wants in one sentence.
Do NOT answer the question.
Do NOT add biomedical facts.

Examples:
Q: What are targets for diabetes?
A: The user wants to identify disease-associated proteins.

Q: What proteins bind this molecule?
A: The user wants to identify protein targets of a molecule.

Query:
{query}
"""
    return call_ollama("llama3:8b", system, user)


def optimize_query(query):
    system = "You extract result size constraints from user queries."
    user = f"""
Extract result size constraint and return JSON only.

Rules:
- If user asks for top N → {{ "limit": N }}
- If user asks for all → {{ "limit": "ALL" }}
- If no number mentioned → {{ "limit": 10 }}

Examples:
Q: top 5 targets for diabetes
A: {{ "limit": 5 }}

Q: show all pathways involved in cancer
A: {{ "limit": "ALL" }}

Q: what are targets for Type 2 Diabetes?
A: {{ "limit": 10 }}

Query:
{query}

Return JSON only.
"""
    raw = call_ollama("llama3:8b", system, user)
    print(" Raw optimization output:", raw)
    return safe_json_load(raw)


def detect_intent(query):
    system = "You classify biomedical query intent."
    user = f"""
Classify the intent into ONE label.

Phase 1:
- DISEASE_TO_TARGET
- DISEASE_TO_PATHWAY
- DRUG_TO_TARGET
- MOLECULE_TO_TARGET
- TARGET_TO_PATHWAY
- COMMON_DISEASE_DISEASE_PATHWAY
- COMMON_DISEASE_PROTEIN_PATHWAY
- COMMON_PROTEIN_PROTEIN_PATHWAY
- DISEASE_PPI_NETWORK
- DISEASE_PPI_HUBS
- MOLECULE_TO_TARGET
- MOLECULE_TO_PATHWAY
- MOLECULE_TO_DISEASE
- PATHWAY_HUB_ANALYSIS
- PATHWAY_DISEASE_BURDEN
- PATHWAY_TO_DISEASES
- CROSS_DISEASE_PATHWAYS
- PATHWAY_LITERATURE_VALIDATION

Phase 2 (Structural):
- TARGET_STRUCTURE
- TARGET_POCKET
- TARGET_DRUGGABILITY
- TARGET_LIGAND
- TARGET_DOCKING
- TARGET_VIRTUAL_SCREENING

Phase 3 (Integrated):
- DISEASE_DRUGGABLE_TARGETS
- FULL_PIPELINE
- TARGET_TO_LIGAND_DISCOVERY

Examples:
Q: What are targets for Alzheimer's disease?
A: DISEASE_TO_TARGET

Q: Which pathways are involved in Parkinson's disease?
A: DISEASE_TO_PATHWAY

Q: What proteins does Metformin act on?
A: DRUG_TO_TARGET

Q: What proteins bind this molecule O=C1Nc2ccccc2?
A: MOLECULE_TO_TARGET

Q: Common pathways between disease A and disease B
A: COMMON_DISEASE_DISEASE_PATHWAY

Q: Which proteins form the interaction network for Alzheimer disease 18?
A: DISEASE_PPI_NETWORK

Q: Which proteins are central or hub nodes in the Alzheimer disease 18 PPI network?
A: DISEASE_PPI_HUBS

Q: Which pathways does <TARGET> or participate in?
A: TARGET_TO_PATHWAY

Q: Pathways involving AGRN
A: TARGET_TO_PATHWAY

Q: What pathways is protein ACTN1 involved in?
A: TARGET_TO_PATHWAY

Q: Which pathways are affected by metformin?
A: MOLECULE_TO_PATHWAY

Q: Which diseases are associated with aspirin targets?
A: MOLECULE_TO_DISEASE

Q: Identify druggable targets in Alzheimer's disease
A: DISEASE_DRUGGABLE_TARGETS

Q: Find ligands for P07471
A: TARGET_LIGAND

Q: Perform docking for protein P07471
A: TARGET_DOCKING

Q: Run full drug discovery pipeline for lung cancer
A: FULL_PIPELINE

Q: Find drug candidates for EGFR
A: TARGET_TO_LIGAND_DISCOVERY

Q: Which pathways are shared between lung cancer and breast cancer?
A: CROSS_DISEASE_PATHWAYS

Q: What diseases share the MAPK signaling pathway?
A: PATHWAY_TO_DISEASES

Q: Which pathways act as hubs connecting most diseases?
A: PATHWAY_HUB_ANALYSIS

Q: How many diseases does the PI3K pathway affect?
A: PATHWAY_DISEASE_BURDEN

Q: Validate the pathways found in lung cancer with literature
A: PATHWAY_LITERATURE_VALIDATION

Q: What is the literature evidence for PI3K pathway in lung cancer?
A: PATHWAY_LITERATURE_VALIDATION

Q: Which pathways are common across diabetes, obesity, and cancer?
A: CROSS_DISEASE_PATHWAYS

Query:
{query}

Return ONLY the intent label.
"""
    return call_ollama("llama3:8b", system, user).strip()


print("✅ Query understanding functions loaded")

✅ Query understanding functions loaded


In [6]:
# ─── Entity Extraction ───
UNIPROT_PATTERN = re.compile(r"\b[OPQ][0-9][A-Z0-9]{3}[0-9]\b")
SMILES_PATTERN  = re.compile(r"[A-Za-z0-9@+\-\[\]\(\)=#$]{6,}")

def extract_uniprot(query):
    match = UNIPROT_PATTERN.search(query)
    return match.group() if match else None


def extract_entities(query):
    uniprot_match = UNIPROT_PATTERN.search(query)
    if uniprot_match:
        return {"Disease": [], "Drug": [], "Protein": [uniprot_match.group()], "Molecule": [], "Pathway": []}

    smiles_match = SMILES_PATTERN.search(query)
    if smiles_match and "=" in smiles_match.group():
        return {"Disease": [], "Drug": [], "Protein": [], "Molecule": [smiles_match.group()], "Pathway": []}

    system = "You extract biomedical entities from user queries."
    user = f"""
Extract biomedical entities and return JSON only.

Allowed entity types:
- Disease
- Drug
- Protein
- Molecule
- Pathway

Rules:
- Molecule can be a chemical name OR SMILES string
- Drug names go under Drug
- If unsure between Drug and Molecule → prefer Molecule
- Do NOT invent entities
- Do NOT infer missing entities
- Return empty lists if not present
- Preserve original spelling

Examples:
Q: Predict pockets for EGFR
A: {{"Disease":[],"Drug":[],"Protein":["EGFR"],"Molecule":[],"Pathway":[]}}

Q: Identify druggable targets in Type 2 Diabetes
A: {{"Disease":["Type 2 Diabetes"],"Drug":[],"Protein":[],"Molecule":[],"Pathway":[]}}

Query:
{query}

Return JSON only.
"""
    raw = call_ollama("llama3:8b", system, user)
    print("Raw entity output:", raw)
    return safe_json_load(raw)


def correct_query_spelling_and_entities(query):
    system = "You correct spelling and normalize biomedical entity names."
    user = f"""
Correct spelling mistakes and minor grammar errors.
Return only the corrected query.

Rules:
- Do NOT change the intent
- NO semantic rewriting
- ONLY spelling / grammar
- Preserve the user's meaning
- DO NOT change protein IDs
- Keep SMILES strings unchanged
- DO NOT replace diseases with more specific or different diseases
- DO NOT add or remove words
- DO NOT explain corrections

Examples:
Type 2 diabates → Type 2 diabetes mellitus
Alzeimer disease → Alzheimer disease 18
aplpha 2 plasmin inhibitor deficiency → Alpha-2-plasmin inhibitor deficiency
lung canser → lung cancer
P07471 → P07471

DO NOT DO:
lung cancer → lung carcinoma
breast tumor → breast carcinoma

Query:
{query}
"""
    return call_ollama("llama3:8b", system, user).strip()


print("✅ Entity extraction functions loaded")

✅ Entity extraction functions loaded


In [7]:
# ─── Entity Resolution ───
from rapidfuzz import process, fuzz

def normalize_text_loose(text):
    return re.sub(r'[^a-z0-9]', '', text.lower())

def match_confidence(score):
    if score > 85:   return "high"
    elif score > 65: return "medium"
    return "low"

def normalize_key(text):
    return normalize_text_loose(text) if text else ""

def select_primary_disease(disease_mentions):
    for d in disease_mentions:
        if "deficiency" in d.lower(): return d
    for d in disease_mentions:
        if "mutation" in d.lower(): return d
    return max(disease_mentions, key=len)


def resolve_disease_kg_grounded(query, min_score=40):
    disease_nodes = [
        (node, data.get("disease_name", node))
        for node, data in knowledge_graph.nodes(data=True)
        if data.get("entity_type") == "disease"
    ]
    if not disease_nodes: return None
    disease_names = [name for _, name in disease_nodes]
    match, score, idx = process.extractOne(query, disease_names, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    return {"disease_id": disease_nodes[idx][0], "disease_name": disease_nodes[idx][1],
            "match_score": score, "confidence": "high" if score > 85 else "medium"}


def resolve_protein_kg_grounded(protein_text, min_score=40):
    query_raw  = protein_text.strip()
    query_norm = normalize_text_loose(protein_text)
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            if (node.lower() == query_raw.lower() or
                data.get("protein_name", "").lower() == query_raw.lower() or
                data.get("gene_name", "").lower() == query_raw.lower()):
                return {"protein_id": node, "protein_name": data.get("protein_name", node),
                        "gene_name": data.get("gene_name", ""), "match_score": 100, "confidence": "high"}
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            combined = f"{node} {data.get('protein_name','')} {data.get('gene_name','')}"
            if normalize_text_loose(combined) == query_norm:
                return {"protein_id": node, "protein_name": data.get("protein_name", node),
                        "gene_name": data.get("gene_name", ""), "match_score": 95, "confidence": "high"}
    candidates, texts = [], []
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "protein":
            combined = f"{node} {data.get('protein_name','')} {data.get('gene_name','')}"
            candidates.append((node, data)); texts.append(normalize_text_loose(combined))
    if not texts: return None
    match, score, idx = process.extractOne(query_norm, texts, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    node_id, node_data = candidates[idx]
    return {"protein_id": node_id, "protein_name": node_data.get("protein_name", node_id),
            "gene_name": node_data.get("gene_name", ""), "match_score": score, "confidence": match_confidence(score)}


def resolve_pathway_kg_grounded(query, min_score=40):
    if not query: return None
    query_norm = query.lower().strip()
    pathway_nodes = [(node, data.get("pathway_name", "").lower())
                     for node, data in knowledge_graph.nodes(data=True) if data.get("entity_type") == "pathway"]
    if not pathway_nodes: return None
    choices = [name for _, name in pathway_nodes]
    match, score, idx = process.extractOne(query_norm, choices, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    matched_node = pathway_nodes[idx][0]
    node_data = knowledge_graph.nodes[matched_node]
    return {"pathway_id": matched_node, "pathway_name": node_data.get("pathway_name", ""),
            "source": node_data.get("source", ""), "match_score": score}


def resolve_drug_kg_grounded(drug_text, min_score=40):
    drug_text = drug_text.lower()
    best_match, best_score = None, 0
    for node_id, node_data in knowledge_graph.nodes(data=True):
        drug_name = node_data.get("drug_name") or node_data.get("name")
        if not drug_name: continue
        score = fuzz.partial_ratio(drug_text, drug_name.lower())
        if score > best_score: best_score = score; best_match = (node_id, node_data)
    if best_match and best_score >= min_score:
        node_id, node_data = best_match
        return {"drug_id": node_id, "drug_name": node_data.get("drug_name") or node_data.get("name"),
                "match_score": best_score, "confidence": match_confidence(best_score)}
    return None


def resolve_molecule_kg_grounded(molecule_text, min_score=60):
    query_norm = normalize_text_loose(molecule_text)
    candidates, texts = [], []
    for node, data in knowledge_graph.nodes(data=True):
        if data.get("entity_type") == "molecule":
            name   = data.get("molecule_name", "")
            smiles = data.get("smiles", "")
            label  = data.get("label", "")
            candidates.append((node, data)); texts.append(normalize_text_loose(f"{name} {smiles} {label}"))
    if not texts: return None
    match, score, idx = process.extractOne(query_norm, texts, scorer=fuzz.token_sort_ratio)
    if score < min_score: return None
    node_id, node_data = candidates[idx]
    return {"molecule_id": node_id, "molecule_name": node_data.get("molecule_name", node_id),
            "smiles": node_data.get("smiles", ""), "match_score": score, "confidence": match_confidence(score)}


def normalize_entities_kg_grounded(entities):
    normalized = {"Disease": [], "Protein": [], "Drug": [], "Molecule": [], "Pathway": []}
    if entities.get("Disease"):
        resolved = resolve_disease_kg_grounded(entities["Disease"][0])
        if resolved: normalized["Disease"].append(resolved)
    if entities.get("Protein"):
        resolved = resolve_protein_kg_grounded(entities["Protein"][0])
        if resolved: normalized["Protein"].append(resolved)
    if entities.get("Drug"):
        resolved_drug = resolve_drug_kg_grounded(entities["Drug"][0])
        if resolved_drug: normalized["Drug"].append(resolved_drug)
        else:
            resolved_mol = resolve_molecule_kg_grounded(entities["Drug"][0])
            if resolved_mol: normalized["Molecule"].append(resolved_mol)
    if entities.get("Molecule") and not normalized["Drug"]:
        resolved = resolve_molecule_kg_grounded(entities["Molecule"][0])
        if resolved: normalized["Molecule"].append(resolved)
    if entities.get("Pathway"):
        resolved = resolve_pathway_kg_grounded(entities["Pathway"][0])
        if resolved: normalized["Pathway"].append(resolved)
    return normalized


print("✅ Entity resolution functions loaded")

✅ Entity resolution functions loaded


In [8]:
# ─── KG Query Functions ───

def build_disease_ppi_subgraph(disease_id, max_nodes=300):
    G_sub = nx.Graph()
    if disease_id not in knowledge_graph: return G_sub
    disease_proteins = set(nbr for nbr in knowledge_graph.neighbors(disease_id)
                           if knowledge_graph.nodes[nbr].get("entity_type") == "protein")
    if not disease_proteins: return G_sub
    expanded_proteins = set(disease_proteins)
    for p in disease_proteins:
        for nbr in knowledge_graph.neighbors(p):
            if knowledge_graph.nodes[nbr].get("entity_type") == "protein":
                expanded_proteins.add(nbr)
                for nbr2 in knowledge_graph.neighbors(nbr):
                    if knowledge_graph.nodes[nbr2].get("entity_type") == "protein":
                        expanded_proteins.add(nbr2)
    expanded_proteins = list(expanded_proteins)[:max_nodes]
    for p in expanded_proteins:
        G_sub.add_node(p, label=knowledge_graph.nodes[p].get("protein_name", p))
    VALID_PPI = ["interacts_with", "binds", "ppi"]
    for u, v, data in knowledge_graph.edges(data=True):
        if data.get("relation") in VALID_PPI and u in expanded_proteins and v in expanded_proteins:
            G_sub.add_edge(u, v)
    return G_sub


def kg_disease_to_targets_direct(resolved_disease, limit=None, ranking=None):
    disease_id = resolved_disease["disease_id"]
    results = []
    for nbr in knowledge_graph.neighbors(disease_id):
        edge = knowledge_graph[disease_id][nbr]
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "protein":
            results.append({"protein_id": nbr, "protein_name": node.get("protein_name", nbr),
                            "gene_name": node.get("gene_name", ""), "relation": edge.get("relation")})
    if ranking and results:
        G = build_disease_ppi_subgraph(disease_id)
        if G.number_of_nodes() > 0:
            scores = nx.degree_centrality(G) if ranking == "degree" else nx.pagerank(G) if ranking == "pagerank" else {}
            for r in results: r["score"] = round(scores.get(r["protein_id"], 0), 4)
            results = sorted(results, key=lambda x: x.get("score", 0), reverse=True)
    if isinstance(limit, int): results = results[:limit]
    return {"disease": resolved_disease, "targets": results, "ranking_method": ranking if ranking else "none"}


def kg_disease_to_pathways(resolved_disease, limit=None):
    disease_id = resolved_disease["disease_id"]
    pathways = {}
    for protein in knowledge_graph.neighbors(disease_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein": continue
        for path in knowledge_graph.neighbors(protein):
            node = knowledge_graph.nodes[path]
            if node.get("entity_type") == "pathway":
                pathways[path] = {"pathway_id": path, "pathway_name": node.get("pathway_name", path),
                                  "source": node.get("source", "")}
    results = list(pathways.values())
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return {"disease": resolved_disease, "pathways": results}


def kg_target_to_pathways(protein_id, limit=None):
    results = []
    for nbr in knowledge_graph.neighbors(protein_id):
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "pathway":
            results.append({"pathway_id": nbr, "pathway_name": node.get("pathway_name", nbr),
                            "source": node.get("source", "")})
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return results


def kg_disease_ppi_network(disease):
    disease_id = disease["disease_id"]
    disease_proteins = set(nbr for nbr in knowledge_graph.neighbors(disease_id)
                           if knowledge_graph.nodes[nbr].get("entity_type") == "protein")
    if not disease_proteins: return {"proteins": [], "edges": []}
    ppi_edges, all_proteins = [], set(disease_proteins)
    for u, v, data in knowledge_graph.edges(data=True):
        if data.get("relation") == "interacts_with" and (u in disease_proteins or v in disease_proteins):
            ppi_edges.append((u, v))
            for n in [u, v]:
                if knowledge_graph.nodes[n].get("entity_type") == "protein": all_proteins.add(n)
    return {"proteins": list(all_proteins), "edges": ppi_edges}


def kg_disease_ppi_hubs(disease, top_k=10):
    G_sub = build_disease_ppi_subgraph(disease["disease_id"])
    if G_sub.number_of_nodes() == 0 or G_sub.number_of_edges() == 0:
        return [{"message": "No PPI data available for this disease"}]
    degree = nx.degree_centrality(G_sub)
    hubs   = sorted(degree.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"protein_id": node, "score": score} for node, score in hubs]


def kg_drug_to_targets(drug_id, limit=None):
    targets = []
    if drug_id not in knowledge_graph: return targets
    for nbr in knowledge_graph.neighbors(drug_id):
        edge = knowledge_graph[drug_id][nbr]
        if edge.get("relation") == "targets":
            pdata = knowledge_graph.nodes[nbr]
            targets.append({"protein_id": nbr, "protein_name": pdata.get("protein_name", nbr),
                            "gene_name": pdata.get("gene_name", ""),
                            "affinity_value": edge.get("affinity_value"),
                            "confidence_score": edge.get("confidence_score")})
    if limit not in [None, "ALL"]: targets = targets[:int(limit)]
    return targets


def kg_molecule_to_targets(molecule_node_id, limit=None):
    targets = {}
    if molecule_node_id not in knowledge_graph: return []
    for nbr in knowledge_graph.neighbors(molecule_node_id):
        edge = knowledge_graph[molecule_node_id][nbr]
        if edge.get("relation") == "has_structure":
            drug = nbr
            for prot in knowledge_graph.neighbors(drug):
                edge2 = knowledge_graph[drug][prot]
                if edge2.get("relation") == "targets":
                    pdata = knowledge_graph.nodes[prot]
                    targets[prot] = {"protein_id": prot, "protein_name": pdata.get("protein_name", prot),
                                     "gene_name": pdata.get("gene_name", ""), "drug_id": drug}
    results = list(targets.values())
    if limit not in [None, "ALL"]: results = results[:int(limit)]
    return results


print("✅ KG query functions loaded")


# ══════════════════════════════════════════════════════════════
# TARGET PRIORITIZATION — 2 Research-Valid Methods
# (No graph metrics — independent of KG completeness)
# ══════════════════════════════════════════════════════════════

import time as _time

def _safe_api(url, params=None, retries=3, delay=2):
    for i in range(retries):
        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code == 200:
                return r.json()
        except Exception as e:
            pass
        _time.sleep(delay)
    return None


def get_structural_druggability_score(uniprot_id: str) -> dict:
    """
    RCSB PDB structural druggability score.
    Score: 4=X-ray+drug ligand, 3=X-ray+ligand, 2=X-ray only, 1=AlphaFold, 0=none
    Reference: Hopkins & Groom, Nat Rev Drug Discov 2002
    """
    BAD_LIGANDS = {"HOH","NA","CL","K","MG","CA","ZN","SO4","PO4",
                   "GOL","EDO","PEG","DMS","MPD","FMT","ACT","IOD"}
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query = {
        "query": {"type": "terminal", "service": "text",
                  "parameters": {
                      "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                      "operator": "exact_match", "value": uniprot_id}},
        "return_type": "entry", "request_options": {"return_all_hits": True}
    }
    try:
        resp = requests.post(search_url, json=query, timeout=15)
        if resp.status_code != 200:
            return {"druggability_score": 0, "pdb_count": 0, "best_resolution": None, "has_drug_ligand": False}
        pdb_ids = [r["identifier"] for r in resp.json().get("result_set", [])]
    except:
        return {"druggability_score": 0, "pdb_count": 0, "best_resolution": None, "has_drug_ligand": False}

    if not pdb_ids:
        try:
            af = requests.head(f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb", timeout=10)
            if af.status_code == 200:
                return {"druggability_score": 1, "pdb_count": 0, "best_resolution": None, "has_drug_ligand": False}
        except:
            pass
        return {"druggability_score": 0, "pdb_count": 0, "best_resolution": None, "has_drug_ligand": False}

    best_res = 99.0; has_drug_lig = False; has_any_lig = False; has_xray = False
    for pdb_id in pdb_ids[:15]:
        entry = _safe_api(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
        if not entry: continue
        try:
            if "X-RAY" in entry["rcsb_entry_info"]["experimental_method"].upper():
                has_xray = True
        except: pass
        try:
            res = entry["rcsb_entry_info"]["resolution_combined"][0]
            if res < best_res: best_res = res
        except: pass
        try:
            ligs = entry["rcsb_entry_container_identifiers"]["non_polymer_entity_ids"]
            if [l for l in ligs if l not in BAD_LIGANDS]:
                has_any_lig = True; has_drug_lig = True
        except: pass

    if has_xray and has_drug_lig:  score = 4
    elif has_xray and has_any_lig: score = 3
    elif has_xray:                 score = 2
    else:                          score = 1

    return {
        "druggability_score": score,
        "pdb_count":          len(pdb_ids),
        "best_resolution":    round(best_res, 2) if best_res < 99 else None,
        "has_drug_ligand":    has_drug_lig
    }


def get_pubmed_literature_score(gene_name: str, disease_name: str) -> dict:
    """
    PubMed co-publication count for gene + disease.
    Score: 4=>100, 3=21-100, 2=6-20, 1=1-5, 0=none
    Reference: Pletscher-Frankild et al., Methods 2015
    """
    if not gene_name or not disease_name:
        return {"pub_count": 0, "literature_score": 0}
    params = {
        "db": "pubmed",
        "term": f'("{gene_name}"[Title/Abstract]) AND ("{disease_name}"[Title/Abstract])',
        "retmode": "json", "retmax": 0,
        "tool": "biomedical_agent", "email": "research@example.com"
    }
    data = _safe_api("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi", params=params)
    count = 0
    try: count = int(data["esearchresult"]["count"])
    except: pass
    if count == 0:      score = 0
    elif count <= 5:    score = 1
    elif count <= 20:   score = 2
    elif count <= 100:  score = 3
    else:               score = 4
    return {"pub_count": count, "literature_score": score}


def prioritize_targets_all_methods(disease: dict, targets: list = None, top_k: int = 10) -> dict:
    """
    Prioritize targets using 2 research-valid independent methods:
    Method 3: Structural Druggability (RCSB PDB)
    Method 4: Literature Evidence (PubMed)
    Consensus: in top-5 of both methods.
    """
    disease_name = disease.get("disease_name", "")
    if targets is None:
        targets = kg_disease_to_targets_direct(disease)["targets"]

    scored = []
    for t in targets:
        uniprot_id   = t["protein_id"]
        gene_name    = t.get("gene_name", "") or t.get("protein_name", "")
        protein_name = t.get("protein_name", uniprot_id)

        print(f"     Scoring {gene_name} ({uniprot_id})...")
        struct = get_structural_druggability_score(uniprot_id)
        lit    = get_pubmed_literature_score(gene_name, disease_name)
        _time.sleep(0.3)

        scored.append({
            "rank_pdb":          0,
            "rank_pubmed":       0,
            "protein_id":        uniprot_id,
            "gene_name":         gene_name,
            "protein_name":      protein_name,
            "structural_score":  struct["druggability_score"],
            "pdb_count":         struct["pdb_count"],
            "best_resolution":   struct["best_resolution"],
            "has_drug_ligand":   struct["has_drug_ligand"],
            "pubmed_count":      lit["pub_count"],
            "literature_score":  lit["literature_score"],
        })

    # Rank by Method 3 (PDB)
    pdb_ranked = sorted(scored,
                        key=lambda x: (x["structural_score"],
                                       x["pdb_count"],
                                       -(x["best_resolution"] or 99)),
                        reverse=True)
    for rank, t in enumerate(pdb_ranked, 1):
        t["rank_pdb"] = rank

    # Rank by Method 4 (PubMed)
    pubmed_ranked = sorted(scored, key=lambda x: x["pubmed_count"], reverse=True)
    for rank, t in enumerate(pubmed_ranked, 1):
        t["rank_pubmed"] = rank

    # Consensus: top-5 of both
    top_n          = min(5, top_k)
    top_pdb_ids    = {t["protein_id"] for t in pdb_ranked[:top_n]}
    top_pubmed_ids = {t["protein_id"] for t in pubmed_ranked[:top_n]}
    consensus_ids  = top_pdb_ids & top_pubmed_ids
    consensus      = sorted(
        [t for t in scored if t["protein_id"] in consensus_ids],
        key=lambda x: x["rank_pdb"] + x["rank_pubmed"]
    )

    return {
        "method_3_structural_druggability": pdb_ranked[:top_k],
        "method_4_literature_evidence":     pubmed_ranked[:top_k],
        "consensus_targets":                consensus,
    }


print("✅ Target prioritization loaded")
print("   Method 3: Structural Druggability (RCSB PDB)")
print("   Method 4: Literature Evidence     (PubMed)")

# ─────────────────────────────────────────────────────────────
# ENHANCED PATHWAY ANALYSIS FUNCTIONS
# ─────────────────────────────────────────────────────────────

def kg_get_pathway_proteins(pathway_id: str) -> list:
    """Get all proteins that participate in a given pathway."""
    proteins = []
    for nbr in knowledge_graph.neighbors(pathway_id):
        node = knowledge_graph.nodes[nbr]
        if node.get("entity_type") == "protein":
            proteins.append({
                "protein_id":   nbr,
                "protein_name": node.get("protein_name", nbr),
                "gene_name":    node.get("gene_name", "")
            })
    return proteins


def kg_get_disease_pathway_set(disease_id: str) -> set:
    """Get all pathway IDs associated with a disease (via proteins)."""
    pathways = set()
    for protein in knowledge_graph.neighbors(disease_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein":
            continue
        for node in knowledge_graph.neighbors(protein):
            if knowledge_graph.nodes[node].get("entity_type") == "pathway":
                pathways.add(node)
    return pathways


def kg_cross_disease_pathways(diseases: list, limit=None) -> dict:
    """
    Find pathways shared across two or more diseases.
    diseases: list of resolved disease dicts
    Returns pathways with count of how many diseases share them.
    """
    disease_pathway_sets = {}
    for disease in diseases:
        did  = disease["disease_id"]
        name = disease["disease_name"]
        disease_pathway_sets[did] = {
            "name":     name,
            "pathways": kg_get_disease_pathway_set(did)
        }

    # Count pathway occurrence across diseases
    pathway_counts = {}
    for did, info in disease_pathway_sets.items():
        for pw_id in info["pathways"]:
            if pw_id not in pathway_counts:
                pathway_counts[pw_id] = {"diseases": [], "count": 0}
            pathway_counts[pw_id]["diseases"].append(info["name"])
            pathway_counts[pw_id]["count"] += 1

    # Keep only pathways shared by ≥2 diseases
    shared = {
        pw_id: info
        for pw_id, info in pathway_counts.items()
        if info["count"] >= 2
    }

    # Build result list
    results = []
    for pw_id, info in shared.items():
        node_data = knowledge_graph.nodes.get(pw_id, {})
        results.append({
            "pathway_id":    pw_id,
            "pathway_name":  node_data.get("pathway_name", pw_id),
            "source":        node_data.get("source", ""),
            "shared_count":  info["count"],
            "shared_diseases": info["diseases"]
        })

    results = sorted(results, key=lambda x: x["shared_count"], reverse=True)
    if limit not in [None, "ALL"]:
        results = results[:int(limit)]

    return {
        "diseases":         [d["disease_name"] for d in diseases],
        "shared_pathways":  results,
        "total_shared":     len(results)
    }


def kg_pathway_to_diseases(pathway: dict, limit=None) -> dict:
    """
    Find all diseases associated with a specific pathway.
    """
    pathway_id = pathway["pathway_id"]
    diseases   = {}

    # pathway → proteins → diseases
    for protein in knowledge_graph.neighbors(pathway_id):
        if knowledge_graph.nodes[protein].get("entity_type") != "protein":
            continue
        for disease in knowledge_graph.neighbors(protein):
            node = knowledge_graph.nodes[disease]
            if node.get("entity_type") == "disease":
                diseases[disease] = {
                    "disease_id":   disease,
                    "disease_name": node.get("disease_name", disease)
                }

    results = list(diseases.values())
    if limit not in [None, "ALL"]:
        results = results[:int(limit)]

    return {
        "pathway":  pathway,
        "diseases": results,
        "count":    len(results)
    }


def kg_pathway_hub_analysis(top_k: int = 20) -> dict:
    """
    Find pathway hubs — pathways connected to the most diseases.
    A pathway hub is biologically central and likely involved in
    multiple disease mechanisms.
    """
    pathway_nodes = [
        (node, data)
        for node, data in knowledge_graph.nodes(data=True)
        if data.get("entity_type") == "pathway"
    ]

    hub_scores = []
    for pw_id, pw_data in pathway_nodes:
        # Count unique diseases this pathway connects to
        diseases_connected = set()
        proteins_connected = set()

        for nbr in knowledge_graph.neighbors(pw_id):
            if knowledge_graph.nodes[nbr].get("entity_type") == "protein":
                proteins_connected.add(nbr)
                for nbr2 in knowledge_graph.neighbors(nbr):
                    if knowledge_graph.nodes[nbr2].get("entity_type") == "disease":
                        diseases_connected.add(nbr2)

        if diseases_connected:
            hub_scores.append({
                "pathway_id":       pw_id,
                "pathway_name":     pw_data.get("pathway_name", pw_id),
                "source":           pw_data.get("source", ""),
                "disease_count":    len(diseases_connected),
                "protein_count":    len(proteins_connected),
                "hub_score":        round(len(diseases_connected) / max(len(proteins_connected), 1), 3)
            })

    hub_scores = sorted(hub_scores, key=lambda x: x["disease_count"], reverse=True)
    return {
        "pathway_hubs":       hub_scores[:top_k],
        "total_pathways_checked": len(pathway_nodes)
    }


def kg_pathway_disease_burden(pathway: dict) -> dict:
    """
    Calculate disease burden of a pathway —
    how many and which diseases are linked to it.
    """
    pathway_id = pathway["pathway_id"]
    proteins   = kg_get_pathway_proteins(pathway_id)

    all_diseases = {}
    for p in proteins:
        pid = p["protein_id"]
        for nbr in knowledge_graph.neighbors(pid):
            node = knowledge_graph.nodes[nbr]
            if node.get("entity_type") == "disease":
                all_diseases[nbr] = node.get("disease_name", nbr)

    return {
        "pathway":         pathway,
        "protein_count":   len(proteins),
        "disease_count":   len(all_diseases),
        "diseases":        [{"disease_id": did, "disease_name": dname}
                            for did, dname in all_diseases.items()]
    }


def kg_pathway_literature_validation(pathway_name: str, disease_name: str = "") -> dict:
    """
    Validate a pathway with PubMed literature evidence.
    Searches PubMed for co-publications of pathway name + disease name.
    Returns publication count and literature score.
    """
    import time

    def pubmed_search(query_term):
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        params = {
            "db": "pubmed", "term": query_term,
            "retmode": "json", "retmax": 0,
            "tool": "biomedical_agent", "email": "research@example.com"
        }
        try:
            resp = requests.get(base_url, params=params, timeout=15)
            if resp.status_code == 200:
                return int(resp.json()["esearchresult"]["count"])
        except:
            pass
        return 0

    # Search 1: pathway alone
    pw_count = pubmed_search(f'"{pathway_name}"[Title/Abstract]')
    time.sleep(0.5)  # NCBI rate limit

    # Search 2: pathway + disease
    combined_count = 0
    if disease_name:
        combined_count = pubmed_search(
            f'("{pathway_name}"[Title/Abstract]) AND ("{disease_name}"[Title/Abstract])'
        )
        time.sleep(0.5)

    # Score
    if combined_count > 100:   lit_score = 4
    elif combined_count > 20:  lit_score = 3
    elif combined_count > 5:   lit_score = 2
    elif combined_count > 0:   lit_score = 1
    else:                      lit_score = 0

    return {
        "pathway_name":       pathway_name,
        "disease_name":       disease_name,
        "pathway_pub_count":  pw_count,
        "combined_pub_count": combined_count,
        "literature_score":   lit_score,
        "validation":         "Strong" if lit_score >= 3 else
                              "Moderate" if lit_score == 2 else
                              "Weak" if lit_score == 1 else "Not validated"
    }


def kg_disease_pathways_with_literature(resolved_disease: dict, limit=None) -> dict:
    """
    Enhanced pathway analysis:
    1. Get all pathways for disease from KG
    2. Validate each with PubMed literature
    3. Return ranked by literature evidence
    """
    import time

    basic = kg_disease_to_pathways(resolved_disease, limit=None)
    pathways = basic["pathways"]

    print(f"   Validating {len(pathways)} pathways with PubMed...")
    disease_name = resolved_disease["disease_name"]

    validated = []
    for i, pw in enumerate(pathways):
        pw_name = pw["pathway_name"]
        lit     = kg_pathway_literature_validation(pw_name, disease_name)
        time.sleep(0.3)  # NCBI rate limit
        validated.append({
            **pw,
            "pathway_pub_count":  lit["pathway_pub_count"],
            "combined_pub_count": lit["combined_pub_count"],
            "literature_score":   lit["literature_score"],
            "validation":         lit["validation"]
        })
        if (i + 1) % 5 == 0:
            print(f"   Validated {i+1}/{len(pathways)} pathways...")

    validated = sorted(validated, key=lambda x: x["literature_score"], reverse=True)

    if limit not in [None, "ALL"]:
        validated = validated[:int(limit)]

    return {
        "disease":            resolved_disease,
        "pathways":           validated,
        "total":              len(validated),
        "strongly_validated": [p for p in validated if p["literature_score"] >= 3],
        "not_validated":      [p for p in validated if p["literature_score"] == 0]
    }


print("✅ Enhanced pathway analysis functions loaded")
print("   kg_cross_disease_pathways()          — shared pathways across diseases")
print("   kg_pathway_to_diseases()             — all diseases for a pathway")
print("   kg_pathway_hub_analysis()            — pathway hubs (most disease-connected)")
print("   kg_pathway_disease_burden()          — disease burden of one pathway")
print("   kg_pathway_literature_validation()   — PubMed evidence for pathway+disease")
print("   kg_disease_pathways_with_literature() — full validated pathway analysis")


✅ KG query functions loaded
✅ Target prioritization loaded
   Method 3: Structural Druggability (RCSB PDB)
   Method 4: Literature Evidence     (PubMed)
✅ Enhanced pathway analysis functions loaded
   kg_cross_disease_pathways()          — shared pathways across diseases
   kg_pathway_to_diseases()             — all diseases for a pathway
   kg_pathway_hub_analysis()            — pathway hubs (most disease-connected)
   kg_pathway_disease_burden()          — disease burden of one pathway
   kg_pathway_literature_validation()   — PubMed evidence for pathway+disease
   kg_disease_pathways_with_literature() — full validated pathway analysis


In [9]:
# ─── Visualization & Explanation ───

def plot_ppi_network_interactive(G_ppi, disease_id, top_targets=None, notebook=False, hub_percent=0.2):
    net = Network(height="750px", width="100%", bgcolor="#ffffff", font_color="black",
                  notebook=notebook, cdn_resources="in_line")
    net.force_atlas_2based(); net.toggle_physics(True)
    degree     = dict(G_ppi.degree())
    centrality = nx.degree_centrality(G_ppi)
    top_targets = set(top_targets) if top_targets else set()
    drug_targets = set()
    for node in G_ppi.nodes():
        if node in knowledge_graph:
            for nbr in knowledge_graph.neighbors(node):
                if knowledge_graph[nbr][node].get("relation") == "targets": drug_targets.add(node)
    sorted_deg = sorted(degree.values(), reverse=True)
    hub_threshold = sorted_deg[max(1, int(hub_percent*len(sorted_deg)))] if degree else 0
    for node in G_ppi.nodes():
        protein_name = knowledge_graph.nodes[node].get("protein_name", node)
        role = G_ppi.nodes[node].get("role", "neighbor_protein")
        if node in top_targets:            color = "#ff0000"
        elif node in drug_targets:         color = "#e31a1c"
        elif role == "disease_protein":    color = "#33a02c"
        elif degree.get(node,0) >= hub_threshold: color = "#ff7f00"
        else:                              color = "#1f78b4"
        size  = 15 + centrality.get(node, 0) * 120
        title = f"Protein: {protein_name}\nRole: {role}\nDegree: {degree.get(node,0)}\nCentrality: {centrality.get(node,0):.4f}\nDrug Target: {'Yes' if node in drug_targets else 'No'}"
        net.add_node(node, label=protein_name, color=color, size=size, title=title)
    for u, v in G_ppi.edges():
        net.add_edge(u, v, value=G_ppi[u][v].get("weight", 1), color="#999999")
    return net


def format_targets_with_ranking(results):
    if not isinstance(results, dict): return "\n❌ Invalid results format.\n"
    if "disease" not in results and "targets" not in results: return "\n❌ No disease-target data found.\n"
    disease_name = results.get("disease", {}).get("disease_name", "Unknown Disease")
    text = f"\n🧬 Targets associated with {disease_name}:\n\n"
    targets = results.get("targets", [])
    if not targets: return text + "No targets found.\n"
    for i, t in enumerate(targets, 1):
        gene = t.get("gene_name") or t.get("protein_name") or "Unknown"
        text += f"{i}. {gene} (score: {t['score']})\n" if "score" in t else f"{i}. {gene}\n"
    return text


def format_results_plain_text(results, intent):
    if intent == "DISEASE_TO_TARGET":
        disease = results["disease"]["disease_name"]
        text = f"\n🧬 Targets associated with {disease}:\n\n"
        for i, t in enumerate(results.get("all_targets", []), 1):
            text += f"{i}. {t['gene_name']} ({t['protein_id']})\n"

        ranked = results.get("ranked_targets", {})
        if ranked and "method_3_structural_druggability" in ranked:
            # Table 1: Structural Druggability
            text += "\n\n📊 Table 1 — Structural Druggability Ranking (RCSB PDB)\n"
            text += "   Score: 4=X-ray+ligand  3=X-ray+any ligand  2=X-ray only  1=AlphaFold  0=none\n"
            text += f"   {'Rank':<6}{'UniProt':<12}{'Gene':<12}{'Score':<8}{'PDBs':<8}{'Best Res(Å)':<14}{'Drug Ligand'}\n"
            text += "   " + "-"*68 + "\n"
            for t in ranked.get("method_3_structural_druggability", []):
                res = f"{t['best_resolution']}Å" if t['best_resolution'] else "N/A"
                lig = "Yes" if t['has_drug_ligand'] else "No"
                text += (f"   {t['rank_pdb']:<6}{t['protein_id']:<12}"
                         f"{t['gene_name']:<12}{t['structural_score']:<8}"
                         f"{t['pdb_count']:<8}{res:<14}{lig}\n")

            # Table 2: Literature Evidence
            text += "\n\n📊 Table 2 — Literature Evidence Ranking (PubMed)\n"
            text += "   Score: 4=>100 papers  3=21-100  2=6-20  1=1-5  0=none\n"
            text += f"   {'Rank':<6}{'UniProt':<12}{'Gene':<12}{'Score':<8}{'Publications'}\n"
            text += "   " + "-"*52 + "\n"
            for t in ranked.get("method_4_literature_evidence", []):
                text += (f"   {t['rank_pubmed']:<6}{t['protein_id']:<12}"
                         f"{t['gene_name']:<12}{t['literature_score']:<8}"
                         f"{t['pubmed_count']}\n")

            # Consensus
            consensus = ranked.get("consensus_targets", [])
            text += "\n\n★  Consensus Targets (top-5 in both methods):\n"
            if consensus:
                for j, t in enumerate(consensus, 1):
                    text += (f"   {j}. {t['gene_name']} ({t['protein_id']}) "
                             f"— PDB rank #{t['rank_pdb']}, "
                             f"PubMed rank #{t['rank_pubmed']} "
                             f"({t['pubmed_count']} papers)\n")
            else:
                text += "   No protein in top-5 of both methods\n"
        return text
    elif intent == "DISEASE_TO_PATHWAY":
        disease = results["disease"]["disease_name"]
        text = f"\n🧪 Pathways involved in {disease}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1): text += f"{i}. {p['pathway_name']}\n"
        return text
    elif intent == "TARGET_TO_PATHWAY":
        protein = results.get("protein", {}).get("gene_name", "")
        text = f"\n🧬 Pathways involving {protein}:\n\n"
        for i, p in enumerate(results.get("pathways", []), 1): text += f"{i}. {p['pathway_name']}\n"
        return text
    elif intent == "DISEASE_PPI_NETWORK":
        disease = results["disease"]["disease_name"]
        summary = results.get("ppi_network_summary", {})
        return f"\n🔗 PPI Network for {disease}:\n\nProteins: {summary.get('num_proteins',0)}\nInteractions: {summary.get('num_interactions',0)}\n"
    elif intent == "DISEASE_PPI_HUBS":
        disease = results["disease"]["disease_name"]
        text = f"\n⭐ Hub proteins in {disease}:\n\n"
        for i, p in enumerate(results.get("hub_proteins", []), 1):
            gene = p.get("gene_name", p.get("protein_id", "Unknown"))
            degree = p.get("degree")
            text += f"{i}. {gene} (degree: {degree})\n" if degree is not None else f"{i}. {gene}\n"
        return text
    elif intent == "PATHWAY_LITERATURE_VALIDATION":
        disease = results.get("disease", {}).get("disease_name", "")
        pathways = results.get("pathways", [])
        total    = results.get("total", len(pathways))
        strong   = results.get("strongly_validated", [])
        not_val  = results.get("not_validated", [])
        text  = f"\n📚 Literature-Validated Pathways for {disease}:\n"
        text += f"   Total pathways: {total} | "
        text += f"Strongly validated: {len(strong)} | Not validated: {len(not_val)}\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Pubs':<8}{'Score':<7}{'Validation'}\n"
        text += "   " + "-"*90 + "\n"
        for i, p in enumerate(pathways[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source',''):<12}"
                     f"{p.get('combined_pub_count',0):<8}"
                     f"{p.get('literature_score',0):<7}{p.get('validation','')}\n")
        return text

    elif intent == "CROSS_DISEASE_PATHWAYS":
        diseases = results.get("diseases", [])
        shared   = results.get("shared_pathways", [])
        total    = results.get("total_shared", 0)
        text  = f"\n🔀 Cross-Disease Shared Pathways\n"
        text += f"   Diseases: {', '.join(diseases)}\n"
        text += f"   Total shared pathways: {total}\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Shared by'}\n"
        text += "   " + "-"*80 + "\n"
        for i, p in enumerate(shared[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source',''):<12}"
                     f"{p['shared_count']} diseases\n")
        return text

    elif intent == "PATHWAY_HUB_ANALYSIS":
        hubs  = results.get("pathway_hubs", [])
        total = results.get("total_pathways_checked", 0)
        text  = f"\n⭐ Pathway Hub Analysis ({total} pathways checked)\n\n"
        text += f"   {'Rank':<5}{'Pathway':<50}{'Source':<12}{'Diseases':<10}{'Proteins':<10}{'Hub Score'}\n"
        text += "   " + "-"*95 + "\n"
        for i, p in enumerate(hubs[:20], 1):
            name = p["pathway_name"][:48] + ("..." if len(p["pathway_name"]) > 48 else "")
            text += (f"   {i:<5}{name:<50}{p.get('source',''):<12}"
                     f"{p['disease_count']:<10}{p['protein_count']:<10}"
                     f"{p['hub_score']}\n")
        return text

    elif intent == "PATHWAY_TO_DISEASES":
        pathway  = results.get("pathway", {})
        diseases = results.get("diseases", [])
        count    = results.get("count", 0)
        text  = f"\n🧬 Diseases associated with pathway: {pathway.get('pathway_name','')}\n"
        text += f"   Total diseases: {count}\n\n"
        for i, d in enumerate(diseases[:20], 1):
            text += f"   {i}. {d['disease_name']}\n"
        return text

    elif intent == "PATHWAY_DISEASE_BURDEN":
        pathway  = results.get("pathway", {})
        diseases = results.get("diseases", [])
        text  = f"\n📊 Disease Burden of: {pathway.get('pathway_name','')}\n"
        text += f"   Proteins: {results.get('protein_count',0)} | "
        text += f"Diseases: {results.get('disease_count',0)}\n\n"
        for i, d in enumerate(diseases[:20], 1):
            text += f"   {i}. {d['disease_name']}\n"
        return text

    else:
        return str(results)


def explain_results_with_llm(query, results, intent):
    system = "You are a biomedical research assistant explaining computational biology results."
    common_instruction = """
Instructions:
- Explain ONLY using the provided results (no generic biology)
- For each important entity, explain its role using given data
- If pathway functions are provided, use them explicitly
- If ranking is provided, explain WHY top items are ranked higher
- Identify biological patterns (e.g., MAPK signaling cluster, insulin signaling)

Output:
- 6–8 lines explanation
- Mention specific targets/pathways
- Include reasoning (NOT description)
"""
    if intent == "DISEASE_TO_TARGET":
        targets   = [t["gene_name"] for t in results.get("all_targets", [])]
        ranked    = results.get("ranked_targets", {})
        consensus = ranked.get("consensus_targets", []) if ranked else []
        m3 = [(t["gene_name"], t["protein_id"], t["structural_score"],
               t["pdb_count"], t["best_resolution"])
              for t in ranked.get("method_3_structural_druggability", [])[:5]] if ranked else []
        m4 = [(t["gene_name"], t["protein_id"], t["pubmed_count"])
              for t in ranked.get("method_4_literature_evidence", [])[:5]] if ranked else []
        user = f"""Query: {query}

Disease: {results['disease']['disease_name']}
All KG Targets: {targets}

Method 3 - Structural Druggability Ranking (RCSB PDB):
(gene, uniprot, score 0-4, num_PDBs, best_resolution_angstrom)
{m3}

Method 4 - Literature Evidence Ranking (PubMed):
(gene, uniprot, publication_count)
{m4}

Consensus Targets (top-5 of both methods): {[t["gene_name"] for t in consensus]}

{common_instruction}"""
    elif intent == "DISEASE_TO_PATHWAY":
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = f"Query: {query}\n\nDisease: {results['disease']['disease_name']}\n\nPathways:\n{pathways}\n\n{common_instruction}"
    elif intent == "TARGET_TO_PATHWAY":
        protein  = results.get("protein", {}).get("gene_name", "")
        pathways = [p["pathway_name"] for p in results.get("pathways", [])]
        user = f"Query: {query}\n\nProtein: {protein}\n\nPathways:\n{pathways}\n\n{common_instruction}"
    elif intent == "DISEASE_PPI_NETWORK":
        user = f"Query: {query}\n\nNetwork summary:\n{results.get('ppi_network_summary',{})}\n\nProteins:\n{results.get('ppi_network_proteins',[])}\n\n{common_instruction}"
    elif intent == "DISEASE_PPI_HUBS":
        user = f"Query: {query}\n\nHub proteins:\n{results.get('hub_proteins',[])}\n\n{common_instruction}"
    elif intent == "PATHWAY_LITERATURE_VALIDATION":
        pathways = [(p["pathway_name"], p.get("combined_pub_count",0), p.get("validation",""))
                    for p in results.get("pathways", [])[:10]]
        user = f"""Query: {query}

Disease: {results.get("disease", {}).get("disease_name", "")}
Validated Pathways (name, publication_count, validation_strength):
{pathways}

{common_instruction}"""

    elif intent == "CROSS_DISEASE_PATHWAYS":
        shared = [(p["pathway_name"], p["shared_count"], p["shared_diseases"])
                  for p in results.get("shared_pathways", [])[:10]]
        user = f"""Query: {query}

Diseases compared: {results.get("diseases", [])}
Shared pathways (name, shared_by_N_diseases, which_diseases):
{shared}

{common_instruction}"""

    elif intent == "PATHWAY_HUB_ANALYSIS":
        hubs = [(p["pathway_name"], p["disease_count"], p["protein_count"], p["hub_score"])
                for p in results.get("pathway_hubs", [])[:10]]
        user = f"""Query: {query}

Top pathway hubs (name, disease_count, protein_count, hub_score):
{hubs}

{common_instruction}"""

    elif intent in ["PATHWAY_TO_DISEASES", "PATHWAY_DISEASE_BURDEN"]:
        user = f"""Query: {query}

Pathway: {results.get("pathway", {}).get("pathway_name", "")}
Diseases connected: {[d["disease_name"] for d in results.get("diseases", [])[:15]]}

{common_instruction}"""

    else:
        user = f"Query: {query}\n\nResults:\n{results}\n\n{common_instruction}"
    return call_ollama("llama3:8b", system, user)


print("✅ Visualization & explanation functions loaded")


print("✅ Pathway analysis functions loaded")


✅ Visualization & explanation functions loaded
✅ Pathway analysis functions loaded


In [10]:
# ─── Original run_pipeline() — COMPLETELY UNCHANGED ───

def run_pipeline(query):

    print("\n========== PIPELINE START ==========\n")

    # 0. Spelling correction
    query = correct_query_spelling_and_entities(query)
    print("Corrected Query:", query)

    # 1. Query understanding
    print("\n--- Query Understanding ---")
    print(understand_query(query))

    # 2. Intent
    intent = detect_intent(query)
    print("\n--- Intent ---"); print(intent)

    # 3. Entity extraction
    entities = extract_entities(query)
    print("\n--- Entities ---"); print(entities)

    # 4. Normalize
    normalized = normalize_entities_kg_grounded(entities)
    normalized["Protein"] = [p for p in normalized.get("Protein", [])
                              if p.get("confidence") == "high" or p.get("match_score", 0) >= 80]
    print("\n--- Normalized ---"); print(normalized)

    # 5. Query optimization
    constraints = optimize_query(query)
    limit = constraints.get("limit", 10)
    match = re.search(r'\btop\s*(\d+)', query.lower())
    if match: limit = int(match.group(1))
    print("\nTop results:", limit)

    results = {}

    if intent == "DISEASE_TO_TARGET" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw_targets    = kg_disease_to_targets_direct(disease, limit="ALL")["targets"]
        ranked_targets = prioritize_targets_all_methods(disease, targets=raw_targets, top_k=10)
        results = {"disease": disease, "all_targets": raw_targets, "ranked_targets": ranked_targets}

    elif intent == "DISEASE_TO_PATHWAY" and normalized.get("Disease"):
        results = kg_disease_to_pathways(normalized["Disease"][0], limit)

    elif intent == "TARGET_TO_PATHWAY" and normalized.get("Protein"):
        protein = normalized["Protein"][0]
        results = {"protein": protein, "pathways": kg_target_to_pathways(protein["protein_id"], limit)}

    elif intent == "DISEASE_PPI_NETWORK" and normalized.get("Disease"):
        disease    = normalized["Disease"][0]
        disease_id = disease["disease_id"]
        G_ppi      = build_disease_ppi_subgraph(disease_id)
        if G_ppi.number_of_nodes() > 0:
            print(f"\nRendering simple PPI network for {disease['disease_name']} ...")
            net  = plot_ppi_network_interactive(G_ppi, disease_id=disease_id, notebook=True)
            html = net.generate_html()
            display(HTML(html))
            html_file = f"ppi_{disease_id}.html"
            with open(html_file, "w", encoding="utf-8") as f: f.write(html)
            results = {"disease": disease,
                       "ppi_network_summary": {"num_proteins": G_ppi.number_of_nodes(), "num_interactions": G_ppi.number_of_edges()},
                       "ppi_network_proteins": list(G_ppi.nodes())}
        else:
            print("No PPI interactions available.")
            results = {"disease": disease, "message": "No PPI interactions found"}

    elif intent == "DISEASE_PPI_HUBS" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        results = {"disease": disease, "hub_proteins": kg_disease_ppi_hubs(disease, limit)}

    else:
        results = {"message": "No valid result found."}

    formatted_output = format_results_plain_text(results, intent)
    print("\n--- Results ---"); print(formatted_output)

    explanation = explain_results_with_llm(query, results, intent)
    print("\n--- Explanation ---"); print(explanation)
    print("\n=========== PIPELINE END ===========\n")

    return results, intent, explanation


print("✅ Original run_pipeline() loaded")

✅ Original run_pipeline() loaded


## ─── CELL C: LangGraph Agent (wraps original pipeline) ───

In [11]:
# ══════════════════════════════════════════════════════════════
# LANGGRAPH STATE
# ══════════════════════════════════════════════════════════════

class KGState(TypedDict):
    query:            str
    corrected_query:  str
    intent:           str
    entities:         dict
    normalized:       dict
    limit:            Any
    kg_results:       dict
    formatted_text:   str
    explanation:      str
    druggable_targets: list   # top proteins → passed to Obj-2
    messages:         Annotated[list, operator.add]


# ─── Node 1: Preprocess ───
def node_preprocess(state: KGState) -> KGState:
    query      = state["query"]
    corrected  = correct_query_spelling_and_entities(query)
    intent     = detect_intent(corrected)
    entities   = extract_entities(corrected)
    normalized = normalize_entities_kg_grounded(entities)
    normalized["Protein"] = [p for p in normalized.get("Protein", [])
                              if p.get("confidence") == "high" or p.get("match_score", 0) >= 80]
    constraints = optimize_query(corrected)
    limit       = constraints.get("limit", 10)
    m = re.search(r'\btop\s*(\d+)', corrected.lower())
    if m: limit = int(m.group(1))
    understanding = understand_query(corrected)
    print(f"\n🔍 Preprocessed | Intent: {intent} | Understanding: {understanding}")
    return {**state, "corrected_query": corrected, "intent": intent, "entities": entities,
            "normalized": normalized, "limit": limit,
            "messages": [AIMessage(content=f"Intent: {intent}. Understanding: {understanding}")]}


# ─── Node 2: KG Query — mirrors run_pipeline() routing exactly ───
def node_kg_query(state: KGState) -> KGState:
    intent     = state["intent"]
    normalized = state["normalized"]
    limit      = state["limit"]
    results    = {}

    print(f"\n🧬 KG Query | Intent: {intent}")

    if intent == "DISEASE_TO_TARGET" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw_targets = kg_disease_to_targets_direct(disease, limit="ALL")["targets"]
        ranked      = prioritize_targets_all_methods(disease, targets=raw_targets, top_k=10)
        results     = {"disease": disease, "all_targets": raw_targets, "ranked_targets": ranked}

    elif intent == "DISEASE_TO_PATHWAY" and normalized.get("Disease"):
        results = kg_disease_to_pathways(normalized["Disease"][0], limit)

    elif intent == "TARGET_TO_PATHWAY" and normalized.get("Protein"):
        protein = normalized["Protein"][0]
        results = {"protein": protein, "pathways": kg_target_to_pathways(protein["protein_id"], limit)}

    elif intent == "DRUG_TO_TARGET" and normalized.get("Drug"):
        drug    = normalized["Drug"][0]
        targets = kg_drug_to_targets(drug["drug_id"], limit)
        results = {"drug": drug, "targets": targets}

    elif intent == "MOLECULE_TO_TARGET" and normalized.get("Molecule"):
        mol     = normalized["Molecule"][0]
        targets = kg_molecule_to_targets(mol["molecule_id"], limit)
        results = {"molecule": mol, "targets": targets}

    elif intent == "DISEASE_PPI_NETWORK" and normalized.get("Disease"):
        disease    = normalized["Disease"][0]
        disease_id = disease["disease_id"]
        G_ppi      = build_disease_ppi_subgraph(disease_id)
        if G_ppi.number_of_nodes() > 0:
            net  = plot_ppi_network_interactive(G_ppi, disease_id=disease_id, notebook=True)
            html = net.generate_html()
            display(HTML(html))
            with open(f"ppi_{disease_id}.html", "w", encoding="utf-8") as f: f.write(html)
            results = {"disease": disease,
                       "ppi_network_summary": {"num_proteins": G_ppi.number_of_nodes(), "num_interactions": G_ppi.number_of_edges()},
                       "ppi_network_proteins": list(G_ppi.nodes())}
        else:
            results = {"disease": disease, "message": "No PPI interactions found"}

    elif intent == "DISEASE_PPI_HUBS" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        results = {"disease": disease, "hub_proteins": kg_disease_ppi_hubs(disease, int(limit) if isinstance(limit, int) else 10)}

    elif intent in ["DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE"] and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        raw_targets = kg_disease_to_targets_direct(disease, limit="ALL")["targets"]
        ranked      = prioritize_targets_all_methods(disease, targets=raw_targets, top_k=10)
        results     = {"disease": disease, "all_targets": raw_targets, "ranked_targets": ranked}

    elif intent == "PATHWAY_LITERATURE_VALIDATION" and normalized.get("Disease"):
        disease = normalized["Disease"][0]
        print(f"   Fetching and validating pathways for {disease['disease_name']}...")
        results = kg_disease_pathways_with_literature(disease, limit=limit)


    elif intent == "CROSS_DISEASE_PATHWAYS" and normalized.get("Disease"):
        # Extract multiple diseases from the query
        disease1 = normalized["Disease"][0]
        # Try to find a second disease in entities
        diseases = [disease1]
        raw_entities = state.get("entities", {})
        all_disease_mentions = raw_entities.get("Disease", [])
        for dname in all_disease_mentions[1:]:
            resolved2 = resolve_disease_kg_grounded(dname)
            if resolved2:
                diseases.append(resolved2)
        results = kg_cross_disease_pathways(diseases, limit=limit)


    elif intent == "PATHWAY_HUB_ANALYSIS":
        top_k = int(limit) if isinstance(limit, int) else 20
        print(f"   Analysing pathway hubs (top {top_k})...")
        results = kg_pathway_hub_analysis(top_k=top_k)

    elif intent == "PATHWAY_TO_DISEASES" and normalized.get("Pathway"):
        pathway = normalized["Pathway"][0]
        results = kg_pathway_to_diseases(pathway, limit=limit)

    elif intent == "PATHWAY_DISEASE_BURDEN" and normalized.get("Pathway"):
        pathway = normalized["Pathway"][0]
        results = kg_pathway_disease_burden(pathway)

    else:
        results = {"message": "No valid result found. Check entity extraction."}

    formatted = format_results_plain_text(results, intent)
    print(formatted)
    return {**state, "kg_results": results, "formatted_text": formatted,
            "messages": [AIMessage(content=formatted)]}


# ─── Node 3: LLM Explain ───
def node_llm_explain(state: KGState) -> KGState:
    explanation = explain_results_with_llm(state["corrected_query"], state["kg_results"], state["intent"])
    print("\n--- LLM Explanation ---"); print(explanation)
    return {**state, "explanation": explanation, "messages": [AIMessage(content=explanation)]}


# ─── Node 4: Export shared ───
def node_export_shared(state: KGState) -> KGState:
    """
    Write shared/kg_output.json so Obj-2 knows which UniProt IDs to structurally analyse.
    """
    results = state["kg_results"]
    intent  = state["intent"]

    druggable_targets = []
    if intent in ["DISEASE_TO_TARGET", "DISEASE_DRUGGABLE_TARGETS", "FULL_PIPELINE"]:
        ranked    = results.get("ranked_targets", {})
        # Use consensus targets; fallback to method_3 (structural druggability)
        consensus = ranked.get("consensus_targets", [])
        top_pr    = consensus if consensus else ranked.get("method_3_structural_druggability", [])[:3]
        druggable_targets = [{"protein_id": t["protein_id"], "gene_name": t.get("gene_name", ""),
                              "protein_name": t.get("protein_name", "")} for t in top_pr[:3]]

    shared_payload = {
        "query":             state["corrected_query"],
        "intent":            intent,
        "disease":           results.get("disease", {}),
        "druggable_targets": druggable_targets,
        "kg_summary":        state["formatted_text"],
        "llm_explanation":   state["explanation"]
    }
    with open(KG_OUTPUT_FILE, "w") as f:
        json.dump(shared_payload, f, indent=2)
    print(f"\n✅ shared/kg_output.json written | Druggable targets: {[t['gene_name'] for t in druggable_targets]}")
    return {**state, "druggable_targets": druggable_targets,
            "messages": [AIMessage(content=f"KG output exported. Targets for Obj-2: {druggable_targets}")]}


print("✅ LangGraph nodes defined")

✅ LangGraph nodes defined


In [12]:
# ══════════════════════════════════════════════════════════════
# BUILD LANGGRAPH
# Flow: preprocess → kg_query → llm_explain → export_shared → END
# ══════════════════════════════════════════════════════════════

kg_builder = StateGraph(KGState)
kg_builder.add_node("preprocess",    node_preprocess)
kg_builder.add_node("kg_query",      node_kg_query)
kg_builder.add_node("llm_explain",   node_llm_explain)
kg_builder.add_node("export_shared", node_export_shared)

kg_builder.set_entry_point("preprocess")
kg_builder.add_edge("preprocess",    "kg_query")
kg_builder.add_edge("kg_query",      "llm_explain")
kg_builder.add_edge("llm_explain",   "export_shared")
kg_builder.add_edge("export_shared", END)

kg_agent = kg_builder.compile()
print("✅ Obj-1 KG LangGraph compiled:", list(kg_builder.nodes.keys()))

✅ Obj-1 KG LangGraph compiled: ['preprocess', 'kg_query', 'llm_explain', 'export_shared']


## ─── CELL D: Run Function ───

In [13]:
def run_kg_agent(query: str) -> dict:
    """
    Run the Obj-1 KG Agent through LangGraph.
    Writes shared/kg_output.json for the Orchestrator and Obj-2.
    """
    print("\n" + "═"*60)
    print(f"  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel")
    print(f"  Query: {query}")
    print("═"*60)

    initial: KGState = {
        "query": query, "corrected_query": "", "intent": "", "entities": {},
        "normalized": {}, "limit": 10, "kg_results": {}, "formatted_text": "",
        "explanation": "", "druggable_targets": [],
        "messages": [HumanMessage(content=query)]
    }
    final = kg_agent.invoke(initial)
    print("\n" + "═"*60)
    print("  OBJ-1 COMPLETE — shared/kg_output.json ready")
    print("═"*60)
    return final


print("✅ run_kg_agent() ready")

✅ run_kg_agent() ready


## ─── CELL E: Example Queries ───

In [ ]:
# ─── New Pathway Analysis Queries ───

# Literature-validated pathways for a disease
run_kg_agent("Validate pathways in lung cancer with literature evidence")

# Cross-disease shared pathways
run_kg_agent("Which pathways are shared between lung cancer and breast cancer?")

# Pathway hub analysis
run_kg_agent("Which pathways are the biggest hubs connecting the most diseases?")

# Specific pathway → all diseases
run_kg_agent("Which diseases are associated with the PI3K pathway?")

# Disease burden of a specific pathway
run_kg_agent("How many diseases does the MAPK signaling pathway affect?")


In [13]:
run_kg_agent("What are targets for Lung cancer?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: What are targets for Lung cancer?
════════════════════════════════════════════════════════════
Raw entity output: {"Disease":["lung cancer"],"Drug":[],"Protein":[],"Molecule":[],"Pathway":[]}
 Raw optimization output: { "limit": 10 }

🔍 Preprocessed | Intent: DISEASE_TO_TARGET | Understanding: The user wants to identify disease-associated proteins or molecules related to lung cancer.

🧬 KG Query | Intent: DISEASE_TO_TARGET
     Scoring DLEC1 (Q9Y238)...
     Scoring BRAF (P15056)...
     Scoring EGFR (P00533)...
     Scoring ERBB2 (P04626)...
     Scoring SLC67A1 (Q96BI1)...
     Scoring MXRA5 (Q9NR99)...
     Scoring CASP8 (Q14790)...
     Scoring TP53 (P04637)...

🧬 Targets associated with Lung cancer:

1. DLEC1 (Q9Y238)
2. BRAF (P15056)
3. EGFR (P00533)
4. ERBB2 (P04626)
5. SLC67A1 (Q96BI1)
6. MXRA5 (Q9NR99)
7. CASP8 (Q14790)
8. TP53 (P04637)


📊 Table 1 — Structural 

{'query': 'What are targets for Lung cancer?',
 'corrected_query': 'What are targets for lung cancer?',
 'intent': 'DISEASE_TO_TARGET',
 'entities': {'Disease': ['lung cancer'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0242379',
    'disease_name': 'Lung cancer',
    'match_score': 54.54545454545454,
    'confidence': 'medium'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_results': {'disease': {'disease_id': 'C0242379',
   'disease_name': 'Lung cancer',
   'match_score': 54.54545454545454,
   'confidence': 'medium'},
  'all_targets': [{'protein_id': 'Q9Y238',
    'protein_name': 'Deleted in lung and esophageal cancer protein 1',
    'gene_name': 'DLEC1',
    'relation': 'associated_with'},
   {'protein_id': 'P15056',
    'protein_name': 'Serine/threonine-protein kinase B-raf',
    'gene_name': 'BRAF',
    'relation': 'associated_with'},
   {'protein_id': 'P00533',
    'prot

In [14]:
run_kg_agent("Which all pathways involve in Lung cancer?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: Which all pathways involve in Lung cancer?
════════════════════════════════════════════════════════════
Raw entity output: {
"Disease": ["lung cancer"],
"Drug": [],
"Protein": [],
"Molecule": [],
"Pathway": []
}
 Raw optimization output: { "limit": "ALL" }

🔍 Preprocessed | Intent: DISEASE_TO_PATHWAY | Understanding: The user wants to identify the biological pathways that are associated with or implicated in lung cancer.

🧬 KG Query | Intent: DISEASE_TO_PATHWAY

🧪 Pathways involved in Lung cancer:

1. Spry regulation of FGF signaling
2. Frs2-mediated activation
3. ARMS-mediated activation
4. Signalling to p38 via RIT and RIN
5. RAF activation
6. MAP2K and MAPK activation
7. Negative feedback regulation of MAPK pathway
8. Negative regulation of MAPK pathway
9. Signaling by moderate kinase activity BRAF mutants
10. Signaling by high-kinase activity BRAF mutants
11. Signali

{'query': 'Which all pathways involve in Lung cancer?',
 'corrected_query': 'Which all pathways are involved in lung cancer?',
 'intent': 'DISEASE_TO_PATHWAY',
 'entities': {'Disease': ['lung cancer'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0242379',
    'disease_name': 'Lung cancer',
    'match_score': 54.54545454545454,
    'confidence': 'medium'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 'ALL',
 'kg_results': {'disease': {'disease_id': 'C0242379',
   'disease_name': 'Lung cancer',
   'match_score': 54.54545454545454,
   'confidence': 'medium'},
  'pathways': [{'pathway_id': 'R-HSA-1295596',
    'pathway_name': 'Spry regulation of FGF signaling',
    'source': 'Reactome'},
   {'pathway_id': 'R-HSA-170968',
    'pathway_name': 'Frs2-mediated activation',
    'source': 'Reactome'},
   {'pathway_id': 'R-HSA-170984',
    'pathway_name': 'ARMS-mediated activation',
    'source': '

In [15]:
run_kg_agent("Which pathways involve the protein EGFR?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: Which pathways involve the protein EGFR?
════════════════════════════════════════════════════════════
Raw entity output: {
"Disease": [],
"Drug": [],
"Protein": ["EGFR"],
"Molecule": [],
"Pathway": ["involve", "the", "protein", "EGFR"]
}
 Raw optimization output: { "limit": 10 }

🔍 Preprocessed | Intent: TARGET_TO_PATHWAY | Understanding: The user wants to know which biological pathways or signaling cascades are associated with the protein EGFR.

🧬 KG Query | Intent: TARGET_TO_PATHWAY

🧬 Pathways involving Egfr:

1. Signaling by ERBB2
2. Signaling by ERBB4
3. SHC1 events in ERBB2 signaling
4. PI3K events in ERBB4 signaling
5. SHC1 events in ERBB4 signaling
6. Nuclear signaling by ERBB4
7. Downregulation of ERBB4 signaling
8. PIP3 activates AKT signaling
9. Downregulation of ERBB2:ERBB3 signaling
10. Signaling by EGFR


--- LLM Explanation ---
Based on the provided result

{'query': 'Which pathways involve the protein EGFR?',
 'corrected_query': 'Which pathways involve the protein EGFR?',
 'intent': 'TARGET_TO_PATHWAY',
 'entities': {'Disease': [],
  'Drug': [],
  'Protein': ['EGFR'],
  'Molecule': [],
  'Pathway': ['involve', 'the', 'protein', 'EGFR']},
 'normalized': {'Disease': [],
  'Protein': [{'protein_id': 'P04412',
    'protein_name': 'Epidermal growth factor receptor',
    'gene_name': 'Egfr',
    'match_score': 100,
    'confidence': 'high'}],
  'Drug': [],
  'Molecule': [],
  'Pathway': [{'pathway_id': 'R-MMU-74217',
    'pathway_name': 'Purine salvage',
    'source': 'Reactome',
    'match_score': 47.61904761904761}]},
 'limit': 10,
 'kg_results': {'protein': {'protein_id': 'P04412',
   'protein_name': 'Epidermal growth factor receptor',
   'gene_name': 'Egfr',
   'match_score': 100,
   'confidence': 'high'},
  'pathways': [{'pathway_id': 'R-DME-1227986',
    'pathway_name': 'Signaling by ERBB2',
    'source': 'Reactome'},
   {'pathway_id': '

In [13]:
run_kg_agent("Which pathways does target P07471 participate in?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: Which pathways does target P07471 participate in?
════════════════════════════════════════════════════════════
 Raw optimization output: { "limit": 10 }

🔍 Preprocessed | Intent: TARGET_TO_PATHWAY | Understanding: The user wants to know the biological processes or signaling pathways that involve the protein with UniProt ID P07471.

🧬 KG Query | Intent: TARGET_TO_PATHWAY

🧬 Pathways involving COX6A2:

1. TP53 Regulates Metabolic Genes
2. Respiratory electron transport
3. Cytoprotection by HMOX1
4. Complex IV assembly


--- LLM Explanation ---
Based on the provided results, COX6A2 (P07471) participates in several pathways. Notably, it is involved in "Respiratory electron transport", which suggests that COX6A2 plays a role in the process of generating energy for cells through the respiratory chain.

Additionally, COX6A2 is also part of the "Complex IV assembly" pathway, ind

{'query': 'Which pathways does target P07471 participate in?',
 'corrected_query': 'Which pathways does target P07471 participate in?',
 'intent': 'TARGET_TO_PATHWAY',
 'entities': {'Disease': [],
  'Drug': [],
  'Protein': ['P07471'],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [],
  'Protein': [{'protein_id': 'P07471',
    'protein_name': 'Cytochrome c oxidase subunit 6A2, mitochondrial',
    'gene_name': 'COX6A2',
    'match_score': 100,
    'confidence': 'high'}],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_results': {'protein': {'protein_id': 'P07471',
   'protein_name': 'Cytochrome c oxidase subunit 6A2, mitochondrial',
   'gene_name': 'COX6A2',
   'match_score': 100,
   'confidence': 'high'},
  'pathways': [{'pathway_id': 'R-BTA-5628897',
    'pathway_name': 'TP53 Regulates Metabolic Genes',
    'source': 'Reactome'},
   {'pathway_id': 'R-BTA-611105',
    'pathway_name': 'Respiratory electron transport',
    'source': 'Reactome'},
   {

In [14]:
run_kg_agent("What are the all targets for Type 2 diabetes mellitus disease?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: What are the all targets for Type 2 diabetes mellitus disease?
════════════════════════════════════════════════════════════
Raw entity output: {
"Disease": ["Type 2 diabetes mellitus"],
"Drug": [],
"Protein": [],
"Molecule": [],
"Pathway": []
}
 Raw optimization output: { "limit": "ALL" }

🔍 Preprocessed | Intent: DISEASE_TO_TARGET | Understanding: The user wants to identify disease-associated proteins related to Type 2 diabetes mellitus.

🧬 KG Query | Intent: DISEASE_TO_TARGET
     Scoring INSR (P06213)...
     Scoring LIPC (P11150)...
     Scoring TCF7L2 (Q9NQB0)...
     Scoring SLC30A8 (Q8IWU4)...
     Scoring AKT2 (P31751)...
     Scoring SLC16A11 (Q8NCK7)...
     Scoring PAX4 (O43316)...
     Scoring GCK (P35557)...
     Scoring MT-ND1 (P03886)...
     Scoring ENPP1 (P22413)...
     Scoring SLC2A4 (P14672)...
     Scoring HNF1B (P35680)...
     Scoring HNF4A (P41235

{'query': 'What are the all targets for Type 2 diabetes mellitus disease?',
 'corrected_query': 'What are the all targets for Type 2 diabetes mellitus?',
 'intent': 'DISEASE_TO_TARGET',
 'entities': {'Disease': ['Type 2 diabetes mellitus'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0011860',
    'disease_name': 'Type 2 diabetes mellitus',
    'match_score': 100.0,
    'confidence': 'high'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 'ALL',
 'kg_results': {'disease': {'disease_id': 'C0011860',
   'disease_name': 'Type 2 diabetes mellitus',
   'match_score': 100.0,
   'confidence': 'high'},
  'all_targets': [{'protein_id': 'P06213',
    'protein_name': 'Insulin receptor',
    'gene_name': 'INSR',
    'relation': 'associated_with'},
   {'protein_id': 'P11150',
    'protein_name': 'Hepatic triacylglycerol lipase',
    'gene_name': 'LIPC',
    'relation': 'associated_with'},
   {'protein

In [ ]:
# Full pipeline — also exports top targets to shared/kg_output.json for Obj-2
run_kg_agent("Identify druggable targets in Alzheimer's disease")

In [14]:
run_kg_agent("What are targets for Lung cancer?")


════════════════════════════════════════════════════════════
  OBJ-1 KG AGENT  |  Python 3.8.6 / ipykernel
  Query: What are targets for Lung cancer?
════════════════════════════════════════════════════════════
Raw entity output: {"Disease":["lung cancer"],"Drug":[],"Protein":[],"Molecule":[],"Pathway":[]}
 Raw optimization output: { "limit": 10 }

🔍 Preprocessed | Intent: DISEASE_TO_TARGET | Understanding: The user wants to identify disease-associated proteins or molecules related to lung cancer.

🧬 KG Query | Intent: DISEASE_TO_TARGET
     Scoring DLEC1 (Q9Y238)...
     Scoring BRAF (P15056)...
     Scoring EGFR (P00533)...
     Scoring ERBB2 (P04626)...
     Scoring SLC67A1 (Q96BI1)...
     Scoring MXRA5 (Q9NR99)...
     Scoring CASP8 (Q14790)...
     Scoring TP53 (P04637)...

🧬 Targets associated with Lung cancer:

1. DLEC1 (Q9Y238)
2. BRAF (P15056)
3. EGFR (P00533)
4. ERBB2 (P04626)
5. SLC67A1 (Q96BI1)
6. MXRA5 (Q9NR99)
7. CASP8 (Q14790)
8. TP53 (P04637)


📊 Table 1 — Structural 

{'query': 'What are targets for Lung cancer?',
 'corrected_query': 'What are targets for lung cancer?',
 'intent': 'DISEASE_TO_TARGET',
 'entities': {'Disease': ['lung cancer'],
  'Drug': [],
  'Protein': [],
  'Molecule': [],
  'Pathway': []},
 'normalized': {'Disease': [{'disease_id': 'C0242379',
    'disease_name': 'Lung cancer',
    'match_score': 54.54545454545454,
    'confidence': 'medium'}],
  'Protein': [],
  'Drug': [],
  'Molecule': [],
  'Pathway': []},
 'limit': 10,
 'kg_results': {'disease': {'disease_id': 'C0242379',
   'disease_name': 'Lung cancer',
   'match_score': 54.54545454545454,
   'confidence': 'medium'},
  'all_targets': [{'protein_id': 'Q9Y238',
    'protein_name': 'Deleted in lung and esophageal cancer protein 1',
    'gene_name': 'DLEC1',
    'relation': 'associated_with'},
   {'protein_id': 'P15056',
    'protein_name': 'Serine/threonine-protein kinase B-raf',
    'gene_name': 'BRAF',
    'relation': 'associated_with'},
   {'protein_id': 'P00533',
    'prot